# Diffusion 2020 DDPM

# Imports

In [4]:
%matplotlib inline

import random
import os
import time

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from PIL import Image

from tqdm.notebook import tqdm
import logging

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset

from torchvision.utils import make_grid, save_image
from IPython.display import display, clear_output, HTML

In [5]:
import imagegen.ddpm as ddpm
import imagegen.data as data
import imagegen.fileutil as fu
import imagegen.utils as ut
import imagegen.models as models
import imagegen.train as train

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Config

In [ ]:
cfg=dict(
    data=dict(
        raw_data_dirs=["/Users/davidschneider/data/image/celeba/img_align_celeba"],  # TODO: USE FULL!! IT IS SAMPLE
        resized_dir="/Users/davidschneider/data/image/celeba/img_align_celeba_resized",
        num_files=62500,   # 80% of 62,500 is 50,000 - so this will give 50k training files
        resize=(64, 64),
        train_perc=0.8,
        batch_size=20,
    ),
    unet=dict(
        act='elu',
        channels=(12, 12)
    device='cpu',
    epochs=3,
    num_diffusion_timesteps=1000,
    beta_start=0.0001,
    beta_end=0.02,
)

# Create Resized Images

need to do once

In [ ]:
data.resize_and_save(
    input_dir=cfg['data']['raw_data_dirs'][0], 
    output_dir=cfg['data']['resized_dir'], 
    num_files=cfg['data']['num_files'],
    res=cfg['data']['resize'],
    train_perc=cfg['data']['train_perc']
)

In [ ]:
fu.scale_image("/Users/davidschneider/data/image/celeba/img_align_celeba/081316.jpg")

In [ ]:
fu.display_image_grid(folder=os.path.join(cfg['data']['resized_dir'], 'train'), nr=2, nc=8, width=1.2)

# Data Loaders

In [ ]:
train_ds = data.CachedImageDataset(dir=cfg['data']['resized_dir'] + '/train')
val_ds = data.CachedImageDataset(dir=cfg['data']['resized_dir'] + '/val')

In [ ]:
train_dl = DataLoader(dataset=train_ds, batch_size=cfg['data']['batch_size'], shuffle=True)
val_dl = DataLoader(dataset=val_ds, batch_size=cfg['data']['batch_size'], shuffle=False)

# Unet

In [ ]:
unet = models.UNet()

# Noise Schedule

`betas, alphas, alpha_bars`

In [ ]:
betas = np.linspace(cfg['beta_start'], cfg['beta_end'], cfg['num_diffusion_timesteps'], dtype=np.float64)


In [ ]:
plt.plot(betas)

In [ ]:
def forward_variances(betas):
    x1 = betas[0]
    x2 = (1 - betas[1])*x1 + betas[1]
    x3 = (1 - betas[2])*x2 + betas[2]
    x = [x1, x2, x3]
    for b in betas[3:]:
        x.append((1-b)*x[-1] + b)
    return x

#plt.plot(betas, label='betas')
plt.plot(forward_variances(betas), label='foward_var')
#plt.legend()
plt.title("DDPM x_i Variance's given linear betas [0.001, 0.02]")
plt.ylabel("variance")
plt.xlabel("i");

In [ ]:
alphas = 1.0 - betas
alpha_bars = np.ones_like(alphas)
for k in range(1, len(alphas)):
    alpha_bars[k] = alpha_bars[k-1] * alphas[k]

In [ ]:
plt.plot(alpha_bars)

# Train

In [ ]:
for epoch in cfg


for x in train_dl:
    break

In [ ]:
import denoising_diffusion_pytorch as ddp

In [ ]:
mdl = models.DDPMUnet(cfg=cfg)
train.train_ddpm(cfg=cfg, mdl=mdl, train_dl=train_dl)

In [ ]:
img = Image.open("/Users/davidschneider/data/image/celeba/img_align_celeba_sample/000001.jpg")

In [ ]:
img.mode

# Denoising Diffusion Pytorch

In [7]:
from denoising_diffusion_pytorch import Unet, GaussianDiffusion, Trainer

In [8]:
model = Unet(
    dim = 64,
    dim_mults = (1, 2, 4, 8),
    flash_attn = True
)

In [10]:
models.count_parameters(model)

(35711491, 35711491, 136.2285270690918)

In [11]:
diffusion = GaussianDiffusion(
    model,
    image_size = 64,
    timesteps = 1000,           # number of steps
    sampling_timesteps = 250    # number of sampling timesteps (using ddim for faster inference [see citation for ddim paper])
)


In [13]:
diffusion.betas[0:5], diffusion.betas[-5:]

(tensor([0.0003, 0.0003, 0.0003, 0.0003, 0.0003]),
 tensor([0.2022, 0.2520, 0.3351, 0.5014, 0.9990]))

In [ ]:



trainer = Trainer(
    diffusion,
    '/Users/davidschneider/data/image/celeba/img_align_celeba_resized',
    train_batch_size = 32,
    train_lr = 8e-5,
    train_num_steps = 1000,         # total training steps
    gradient_accumulate_every = 2,    # gradient accumulation steps
    ema_decay = 0.995,                # exponential moving average decay
    amp = False,                       # turn on mixed precision
    calculate_fid = False              # whether to calculate fid during training
)

trainer.train()

In [ ]:
ret = diffusion.sample(batch_size=1, return_all_timesteps=True)

In [ ]:
ret.size()

In [ ]:
plt.imshow(ret[0][0].permute(1, 2, 0).cpu().numpy())

In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 1000  # or 200 for safety (in MB)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Assume ret is a tensor of shape [B, C, H, W] with values in [-1, 1]
frames = ret[0]  # shape: [T, C, H, W] or similar

# Preprocess: unnormalize and convert to numpy
images = []
for x in frames:
    x = (x / 2 + 0.5).clamp(0, 1)  # scale from [-1, 1] → [0, 1]
    img = x.permute(1, 2, 0).cpu().numpy()
    images.append(img)

fig, ax = plt.subplots()
im = ax.imshow(images[0])
ax.axis("off")

def update(frame_idx):
    im.set_array(images[frame_idx])
    return [im]

ani = animation.FuncAnimation(fig, update, frames=len(images), interval=10, blit=True)
plt.close()

HTML(ani.to_jshtml())